# 1. Setup & Environment
- 공간 데이터 및 카토그래피 렌더링 라이브러리(GeoPandas, Matplotlib) 로드
- 한글 폰트(NanumGothic) 및 유니코드 마이너스 부호 환경 설정

In [1]:
# 1. 라이브러리 로드 및 환경 설정
from pathlib import Path
import warnings

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

# 폰트 및 유니코드 설정
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

# 2. Configuration & Policy Target Cartographic Palette
- 공간 경계 및 전이 분석 산출물 입출력 디렉터리(`output/maps`) 설정
- 시인성 극대화를 위한 학술 표준 범주형 색상 체계 정의
  * 지속콜드: Deep Red (`#d73027`, 최우선 공급 타깃)
  * 신규악화: Orange (`#f46d43`, 급격한 결핍화 경고)
  * 개선: Green (`#1a9850`, 인프라 확충 효과)
  * 지속핫: Deep Blue (`#2166ac`, 안정적 최상위)
  * 배경: Light Gray (`#f0f0f0`, Not Significant 및 기타)

In [2]:
# 2. 경로 및 지도 스타일 팔레트 정의
BASE_DIR = Path("/mnt/cowork/EV")
BOUNDARY_FP = BASE_DIR / "input/raw/집계구_2016/집계구.shp"
DIR_MAPS = BASE_DIR / "output/maps"
DIR_MAPS.mkdir(parents=True, exist_ok=True)

# 모델 정의
MODELS = ["2SFCA", "Gravity"]
MODEL_LABEL = {
    "2SFCA": "Gaussian 2SFCA",
    "Gravity": "Gravity Model"
}

# 학술 논문용 전이 유형별 시각화 색상 매핑
COLOR_MAP = {
    "지속콜드": "#d73027",              # 진한 빨강 (4개년 연속 소외 지역)
    "신규악화(Hot->Cold)": "#f46d43",   # 주황 (급격한 악화 지역)
    "개선(Cold->Hot)": "#1a9850",       # 녹색 (인프라 확충 성공 지역)
    "지속핫": "#2166ac",                # 진한 파랑 (4개년 연속 우수 지역)
}

BG_COLOR = "#f0f0f0"        # 기타/비유의 배경색
BORDER_COLOR = "#bdbdbd"    # 집계구 경계선 색상
TEXT_COLOR = "#222222"
MUTED_COLOR = "#666666"

print(f">> 시각화 대상 모형: {MODELS}")
print(f">> 맵 저장 디렉터리: {DIR_MAPS}")

>> 시각화 대상 모형: ['2SFCA', 'Gravity']
>> 맵 저장 디렉터리: /mnt/cowork/EV/output/maps


# 3. Boundary & Transition Data Loader
- 서울시 14,979개 집계구 폴리곤(EPSG:5179) 로드 및 전처리
- 신규 표준 산출물(`hotspot_transition_2021_2024_mw.csv`) 우선 로드

In [3]:
# 3. 집계구 경계 및 전이 분석 데이터 로드
gdf = gpd.read_file(BOUNDARY_FP).set_crs(epsg=5179, allow_override=True)
gdf["TOT_REG_CD"] = gdf["TOT_REG_CD"].astype(str)
gdf_seoul = gdf[gdf["TOT_REG_CD"].str.startswith("11")].copy().reset_index(drop=True)
print(f">> 서울시 집계구 경계 로드 완료: 총 {len(gdf_seoul):,}개 폴리곤")

# 전이 분석 결과 데이터 로드 (_mw 우선 탐색)
fp_trans_mw = BASE_DIR / "output/hotspot_transition_2021_2024_mw.csv"
fp_trans_orig = BASE_DIR / "output/hotspot_transition_2021_2024.csv"

if fp_trans_mw.exists():
    fp_trans = fp_trans_mw
    print(f">> 신규 리팩토링 산출물 로드: {fp_trans.name}")
elif fp_trans_orig.exists():
    fp_trans = fp_trans_orig
    print(f">> 기존 원본 산출물 로드: {fp_trans.name}")
else:
    raise FileNotFoundError("전이 분석 결과 CSV 파일을 찾을 수 없습니다.")

df_trans = pd.read_csv(fp_trans, dtype={"oa_code": str})

>> 서울시 집계구 경계 로드 완료: 총 19,153개 폴리곤
>> 신규 리팩토링 산출물 로드: hotspot_transition_2021_2024_mw.csv


# 4. Cartographic Transition Map Generation & Export (2 Models)
- 모형별 독립 맵 생성 및 군집별 지오메트리 디졸브(Dissolve) 고속 렌더링
- dissolve 직후 부동소수점 오차로 남는 미세 seam(스펙클) 제거용 소버퍼(+1m/-1m) 스냅 적용
- 핵심 정책 지역의 시각적 식별성을 위한 레이어 중첩 순서(Z-order) 제어
- 범례 및 학술 메타데이터 배치 후 300 DPI 이미지(`hotspot_transition_{model}_mw.png`) 저장

In [4]:
# 4. 모형별 전이 지도 시각화 및 개별 저장
print("=" * 80)
print("RUNNING: HOTSPOT TRANSITION CARTOGRAPHIC EXPORT")
print("=" * 80)

# 범례 핸들 정의
legend_handles = [
    mpatches.Patch(facecolor=COLOR_MAP["지속콜드"], label="지속 콜드스팟 (2021 Cold → 2024 Cold)"),
    mpatches.Patch(facecolor=COLOR_MAP["신규악화(Hot->Cold)"], label="신규 악화 (2021 Hot → 2024 Cold)"),
    mpatches.Patch(facecolor=COLOR_MAP["개선(Cold->Hot)"], label="개선 (2021 Cold → 2024 Hot)"),
    mpatches.Patch(facecolor=COLOR_MAP["지속핫"], label="지속 핫스팟 (2021 Hot → 2024 Hot)"),
    mpatches.Patch(facecolor=BG_COLOR, edgecolor=BORDER_COLOR, linewidth=0.5, label="기타 (Not Significant / 기타 전이)"),
]

# 레이어 렌더링 순서 (강조할 정책 타깃을 나중에 그려 최상단에 배치)
RENDER_ORDER = ["지속핫", "개선(Cold->Hot)", "신규악화(Hot->Cold)", "지속콜드"]

for model in MODELS:
    fig, ax = plt.subplots(figsize=(10, 10), facecolor="white")
    
    # 해당 모형 데이터 매핑
    sub_df = df_trans[df_trans["model"] == model].set_index("oa_code")
    g = gdf_seoul.copy()
    g["transition"] = g["TOT_REG_CD"].map(sub_df["transition"]).fillna("기타")

    # 1. 기본 배경 렌더링 (기타 전이 및 비유의 지역)
    g.plot(ax=ax, color=BG_COLOR, edgecolor=BORDER_COLOR, linewidth=0.15)

    # 2. 핵심 전이 유형별 디졸브 렌더링 (순차적 오버레이)
    dissolved = g.dissolve(by="transition")
    # 인접 집계구 경계가 부동소수점 정밀도 차이로 완전히 안 붙어서 dissolve가
    # 매끈한 폴리곤 대신 미세하게 끊긴 조각(스펙클)을 남기는 문제 — 작은 버퍼로 스냅
    dissolved["geometry"] = dissolved.geometry.buffer(1).buffer(-1)
    for key in RENDER_ORDER:
        if key in dissolved.index:
            dissolved.loc[[key]].plot(
                ax=ax, 
                facecolor=COLOR_MAP[key], 
                edgecolor="none", 
                alpha=0.9
            )

    ax.set_axis_off()

    # 상단 메인 타이틀 및 서브타이틀
    ax.set_title(
        f"{MODEL_LABEL[model]} — Spatiotemporal Hotspot Transition (2021 vs 2024)",
        fontsize=14, fontweight="bold", color=TEXT_COLOR, pad=12
    )
    
    # 범례 배치 (좌측 하단)
    ax.legend(
        handles=legend_handles, 
        loc="lower left", 
        frameon=False, 
        fontsize=9.5,
        title="Transition Classes",
        title_fontsize=10.5
    )
    
    # 하단 캡션
    fig.text(
        0.5, 0.04, 
        "Weekdays Daytime (11:00-13:00) | Local Getis-Ord Gi* (KNN k=30, p < 0.05) | Seoul OA (2016)", 
        ha="center", fontsize=9, color=MUTED_COLOR
    )

    # 파일 내보내기 (_mw.png)
    out_fp_mw = DIR_MAPS / f"hotspot_transition_{model}_mw.png"
    fig.savefig(out_fp_mw, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [>] {MODEL_LABEL[model]} 지도 저장 완료 -> {out_fp_mw.name}")

RUNNING: HOTSPOT TRANSITION CARTOGRAPHIC EXPORT
  [>] Gaussian 2SFCA 지도 저장 완료 -> hotspot_transition_2SFCA_mw.png
  [>] Gravity Model 지도 저장 완료 -> hotspot_transition_Gravity_mw.png


# 5. Export Files Verification
- 생성된 2개 모형 전이 지도 이미지 파일의 존재 여부 및 용량(MB) 무결성 확인

In [5]:
# 5. 산출물 파일 상태 확인
records = []
for model in MODELS:
    fp = DIR_MAPS / f"hotspot_transition_{model}_mw.png"
    if fp.exists():
        records.append({
            "모형": MODEL_LABEL[model],
            "파일명": fp.name,
            "파일 크기(MB)": f"{fp.stat().st_size / (1024 * 1024):.2f}",
            "저장 경로": str(fp.resolve()),
            "상태": "정상 생성 완료"
        })
    else:
        records.append({
            "모형": MODEL_LABEL[model],
            "파일명": fp.name,
            "파일 크기(MB)": "-",
            "저장 경로": "-",
            "상태": "파일 누락 확인 필요"
        })

df_status = pd.DataFrame(records)
print("=" * 80)
print("             전이 지도 산출물 내보내기 검증 요약표")
print("=" * 80)
display(df_status)

             전이 지도 산출물 내보내기 검증 요약표


,모형,파일명,파일 크기(MB),저장 경로,상태
0,Gaussian 2SFCA,hotspot_transition_2SFCA_mw.png,1.47,/mnt/cowork/EV/output/maps/hotspot_transition_...,정상 생성 완료
1,Gravity Model,hotspot_transition_Gravity_mw.png,1.44,/mnt/cowork/EV/output/maps/hotspot_transition_...,정상 생성 완료
